<!-- ADDED: Kaggle runner for the SR3 noise-schedule ablation. -->
# SR3 noise-schedule ablation on Kaggle

Runs one fine-tuning run (A or B) end to end on a Kaggle GPU (P100 or T4).

1. Upload the `Solutions` folder as a private Kaggle **Dataset** (zip it without `outputs/`, `data/`, `checkpoints/`, `.venv/`) and attach it to this notebook, or set `REPO_URL` to a git remote you can clone.
2. *Settings*: Accelerator **GPU**, Internet **on**, Persistence **Files only** (keeps `outputs/` between sessions).
3. Set `RUN` below, then *Run all*. One run fits in one 12-hour session; use a second session (or a teammate's account) for the other run.
4. Download `/kaggle/working/run<RUN>_outputs.zip` at the end and merge it into `Solutions/outputs/`.

In [ ]:
from pathlib import Path

RUN = "B"  # "A" or "B"
ALSO_RUN0 = RUN == "A"  # Run 0 only needs to be evaluated once
DATASET_DIR = Path("/kaggle/input/sr3-ablation-solutions")  # attached Kaggle Dataset
REPO_URL = ""  # alternatively: a git URL to clone
WORK = Path("/kaggle/working/Solutions")

In [ ]:
import os
import shutil
import subprocess
import sys

if not WORK.is_dir():
    if REPO_URL:
        subprocess.run(["git", "clone", REPO_URL, str(WORK)], check=True)
    else:
        source = next((p.parent for p in DATASET_DIR.rglob("pyproject.toml")), None)
        assert source, f"pyproject.toml not found under {DATASET_DIR}"
        shutil.copytree(source, WORK)
os.chdir(WORK)
print(sys.version)
!nvidia-smi --query-gpu=name,memory.total --format=csv
# torch / torchvision are preinstalled on Kaggle; install only this package and its light deps.
!pip install -q -e . gdown

In [ ]:
# Keep the large data on Kaggle's scratch disk; outputs stay in /kaggle/working.
local_config = Path("configs/local.toml")
local_config.write_text(
    "[paths]\n"
    'data_root = "/kaggle/tmp/sr3_data"\n'
    'checkpoint_dir = "/kaggle/tmp/sr3_checkpoints"\n'
    "[data]\n"
    "num_workers = 2\n"
)
print(local_config.read_text())

In [ ]:
!python -m sr3_ablation download-checkpoint
!python -m sr3_ablation prepare-data
!python -m sr3_ablation verify-schedule --run {RUN}

In [ ]:
# Optional, about 10 minutes: measures speed on this GPU and writes outputs/smoke/timings.json
# !python -m sr3_ablation smoke-test

In [ ]:
if ALSO_RUN0:
    !python -m sr3_ablation run0

In [ ]:
# Fine-tuning: loss.csv, validation.csv, samples/, checkpoints/, schedule.json -> outputs/run{RUN}
!python scripts/finetune.py --run {RUN}

In [ ]:
!python -m sr3_ablation evaluate --run {RUN}
!python -m sr3_ablation trajectories --run {RUN}

In [ ]:
# Package the outputs for download (intermediate epoch checkpoints are left out).
archive_root = Path(f"/kaggle/working/run{RUN}_outputs")
shutil.rmtree(archive_root, ignore_errors=True)
shutil.copytree(
    "outputs", archive_root / "outputs", ignore=shutil.ignore_patterns("epoch*_gen.pth")
)
print(shutil.make_archive(str(archive_root), "zip", archive_root))